# Sink tokens and a rolling cache in NumPy

A model reading an endless stream cannot keep every key. A common policy keeps two groups:
the first $S$ tokens of the stream, which many heads attend to heavily and which are called
attention sinks, and the $W$ most recent tokens. Everything in between is evicted. With RoPE
at the positions in the stream, the offset from the current query back to a sink grows
without limit and soon passes anything the model met in training. StreamingLLM (Xiao et al.,
2023) numbers the tokens by their place in the cache instead: the sinks take positions $0$ to
$S - 1$ and the window tokens take $S, S + 1, \dots$ in order, so no offset passes
$S + W - 1$.

You will write that numbering from scratch, and then a streaming loop that gives the same
outputs while rotating each window key only once.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. `rope` rotates every row of a matrix at its own position, with the
lesson's layout, and `attend` is softmax attention for one query.

In [ ]:
import numpy as np


def rope(X, pos, base=10000.0):
    """Rotate row t of X, shape (n, d), by pos[t] * theta_i on each coordinate pair (2i, 2i + 1)."""
    X = np.asarray(X, dtype=float)
    d = X.shape[1]
    theta = base ** (-2.0 * np.arange(d // 2) / d)
    z = (X[:, 0::2] + 1j * X[:, 1::2]) * np.exp(1j * np.asarray(pos, dtype=float)[:, None] * theta)
    out = np.empty_like(X)
    out[:, 0::2], out[:, 1::2] = z.real, z.imag
    return out


def attend(q, K, V):
    """Softmax attention of the query q over the rows of K and V."""
    s = K @ q / np.sqrt(len(q))
    w = np.exp(s - s.max())
    return w / w.sum() @ V


# A stream of 40 tokens for one head of width 8. The cache keeps S = 2 sinks and the W = 6 newest tokens.
rng = np.random.default_rng(23)
N, d, S, W = 40, 8, 2, 6
Q, K, V = rng.standard_normal((3, N, d))
print(f"{N} tokens, a cache of {S} sinks and {W} recent tokens")

## Part 1 (coding task): The cache and its positions, from scratch

Write `reindexed_step(Q, K, V, t, S, W)`, the output for query `t` once key `t` has entered
the cache.

- While `t < S + W`, nothing has been evicted, and the cache holds tokens `0` to `t`.
- From then on it holds the sinks `0` to `S - 1` and the window `t - W + 1` to `t`, which
  includes token `t` itself.

Number the cached tokens `0, 1, 2, ...` in their order in the stream. Rotate each cached key
at its number and query `t` at the number of its own key, then return `attend` of the
rotated query over the rotated keys and the cached values. Values are not rotated.

In [ ]:
def reindexed_step(Q, K, V, t, S, W):
    """Output for query t, with RoPE positions numbered inside the cache."""
    # TODO
    raise NotImplementedError

In [ ]:
def cached_tokens(t, S, W):
    return np.arange(t + 1) if t < S + W else np.r_[np.arange(S), np.arange(t - W + 1, t + 1)]

def expected_step(t, S, W):
    """Rotate the keys by their offset from the query and leave the query as it is: the same scores."""
    kept = cached_tokens(t, S, W)
    offsets = np.arange(len(kept)) - (len(kept) - 1)
    return attend(Q[t], rope(K[kept], offsets), V[kept])

def stream_positions_step(t, S, W):
    """The same cache with every token at its position in the stream."""
    kept = cached_tokens(t, S, W)
    return attend(rope(Q[t:t + 1], [t])[0], rope(K[kept], kept), V[kept])

got = np.asarray(reindexed_step(Q, K, V, 5, S, W))
assert got.shape == (d,), f"reindexed_step returned shape {got.shape}, expected ({d},)"
assert np.allclose(got, expected_step(5, S, W)), "query 5 is wrong. Nothing has been evicted yet, so the cache is tokens 0 to 5, each once, at positions 0 to 5, the query takes position 5, and the values are not rotated"
assert np.allclose(reindexed_step(Q, K, V, 7, S, W), expected_step(7, S, W)), "query 7: the cache is just full, tokens 0 to 7 at positions 0 to 7"
got8 = reindexed_step(Q, K, V, 8, S, W)
assert not np.allclose(got8, stream_positions_step(8, S, W)), "query 8: the keys kept their positions in the stream. Number the cached tokens by their place in the cache"
assert np.allclose(got8, expected_step(8, S, W)), "query 8: the cache is tokens 0, 1 and 3 to 8, at positions 0 to 7, with the query at 7"
for t in range(N):
    assert np.allclose(reindexed_step(Q, K, V, t, S, W), expected_step(t, S, W)), f"query {t} is wrong"
for s_, w_, t in [(1, 3, 10), (3, 4, 25), (1, 3, 2), (4, 1, 30)]:
    assert np.allclose(reindexed_step(Q, K, V, t, s_, w_), expected_step(t, s_, w_)), f"query {t} with S = {s_} and W = {w_} is wrong"
print("Part 1 passed")

## Part 2 (coding task): Stream while rotating each window key once

Renumbering every step looks expensive, since each eviction moves every window token down
one place. Write `stream(Q, K, V, S, W)`, which processes the `N` tokens in order and returns
the `N` outputs, shape `(N, d)`, equal to `reindexed_step` at every step, under these rules:

- Query `t` is rotated once, at its position `t` in the stream.
- A key that is not a sink is rotated once, at its position in the stream, when it arrives,
  and stored rotated. It is dropped once it leaves the window.
- The `S` sink keys are stored unrotated, and you may rotate them again at every step, at
  whatever positions make the outputs match Part 1.

Use `rope` for every rotation. The check counts the rows it rotates, and allows at most
`2 * N + S * N`.

In [ ]:
def stream(Q, K, V, S, W):
    """Outputs for every query, with the cache numbered as in Part 1 and window keys rotated once."""
    # TODO
    raise NotImplementedError

In [ ]:
rows_rotated = []
setup_rope = rope
def rope(X, pos, base=10000.0):
    X = np.asarray(X, dtype=float)
    rows_rotated.append(X.size // X.shape[-1])
    return setup_rope(X, pos, base)
try:
    out = np.asarray(stream(Q, K, V, S, W))
finally:
    rope = setup_rope

assert out.shape == (N, d), f"stream returned shape {out.shape}, expected ({N}, {d})"
want = np.array([expected_step(t, S, W) for t in range(N)])
in_stream = np.array([stream_positions_step(t, S, W) for t in range(N)])
if not np.allclose(out, want):
    first = int(np.argmax(~np.isclose(out, want).all(axis=1)))
    if np.allclose(out, in_stream):
        raise AssertionError(f"these are the outputs with every token at its stream position (first wrong at query {first}). Once eviction starts, a sink must be rotated so that its offset from the query is its offset inside the cache")
    raise AssertionError(f"the outputs differ from reindexed_step, first at query {first}")
assert sum(rows_rotated) <= 2 * N + S * N, f"rope rotated {sum(rows_rotated)} rows, more than the {2 * N + S * N} allowed: rotate each window key once, when it arrives, and keep it"
for s_, w_ in [(1, 3), (3, 4), (4, 1)]:
    want_sw = np.array([expected_step(t, s_, w_) for t in range(N)])
    assert np.allclose(stream(Q, K, V, s_, w_), want_sw), f"wrong outputs with S = {s_} and W = {w_}"
print(f"rows rotated: {sum(rows_rotated)} (renumbering the whole cache at every step takes {N + sum(len(cached_tokens(t, S, W)) for t in range(N))})")
print("Part 2 passed")

## Solution

### Part 1: The cache and its positions, from scratch

In [ ]:
def reindexed_step(Q, K, V, t, S, W):
    """Output for query t, with RoPE positions numbered inside the cache."""
    if t < S + W:
        kept = np.arange(t + 1)
    else:
        kept = np.r_[np.arange(S), np.arange(t - W + 1, t + 1)]
    slots = np.arange(len(kept))                    # positions inside the cache
    keys = rope(K[kept], slots)
    q = rope(Q[t:t + 1], slots[-1:])[0]             # the query's own key sits in the last slot
    return attend(q, keys, V[kept])

**Why it works.** The cache is the sinks plus the last `W` tokens, or every token before the
first eviction, and `np.arange(len(kept))` numbers them in stream order. The query is the
newest token, so it takes the last number. The check builds the same output another way:
since a score depends only on the offset between query and key, it rotates each key by its
offset from the query and leaves the query unrotated.

### Part 2: Stream while rotating each window key once

In [ ]:
def stream(Q, K, V, S, W):
    """Outputs for every query, with the cache numbered as in Part 1 and window keys rotated once."""
    n = len(Q)
    sink_k, sink_v = [], []          # stored unrotated
    win_k, win_v = [], []            # rotated at their stream positions
    out = []
    for t in range(n):
        if t < S:
            sink_k.append(K[t])
            sink_v.append(V[t])
        else:
            win_k.append(rope(K[t:t + 1], [t])[0])
            win_v.append(V[t])
            if len(win_k) > W:
                win_k.pop(0)
                win_v.pop(0)
        q = rope(Q[t:t + 1], [t])[0]
        keys, values = list(win_k), list(win_v)
        if sink_k:
            # Sink j sits in slot j and the query in the last slot, so their offset
            # inside the cache is last - j. Seen from the query at t, that puts sink j at
            # stream position t - (last - j).
            last = len(sink_k) + len(win_k) - 1
            at = t - (last - np.arange(len(sink_k)))
            keys = list(rope(np.array(sink_k), at)) + keys
            values = sink_v + values
        out.append(attend(q, np.array(keys), np.array(values)))
    return np.array(out)

**Why it works.** The window is a run of consecutive tokens that ends at the query, so the
offset from the query to a window key is the same whether you count in the stream or in the
cache: a key $r$ tokens back sits $r$ slots back. Window keys rotated once at their stream
positions, against a query at its stream position, already give the renumbered scores. Only
the sinks are out of step. In the cache the query is in slot `last` and sink $j$ in slot $j$,
an offset of `last - j`, so rotating sink $j$ at stream position `t - (last - j)` gives the
offset the cache asks for. Before the first eviction `last` equals `t` and the sinks sit at
their own positions.

With $S = 2$ and $W = 6$ the loop rotates 157 rows, against 332 for renumbering the whole
cache at every step. In a model with 4 sinks and a window of 4,092 tokens the difference is
rotating 5 keys a step (the 4 sinks plus the new key) instead of 4,096. StreamingLLM's own code
stores keys before rotation and rotates the whole cache at every step, which gives the same
outputs.